# Advanced 01 — Single Agent vs Multi-Agent Architecture Decisions

**Northstar question:** Why did EU checkout conversion fall after `deploy-1842`, and what should we do?

> Start with the smallest architecture that works. Split only when a measured structural boundary justifies the coordination cost.

This credential-free notebook compares six architectures on the same task, trusted context, and evidence. All timings, tokens, costs, and route outcomes are deterministic fixtures for teaching—not live-model benchmarks. The core lab is read-only and never performs a production write.


## Part 1 — Architecture decision problem

Success means grounded use of health, logs, deployment, customer-impact, and current-runbook evidence; complete required-fact transfer; no tenant or capability violation; and a bounded recommendation. An approval-gated rollback route remains a proposal, not execution authority.

We import `policy.py` and `lab.py`, the same Pydantic v2 implementation used by pytest. Hard controls are application-owned; model text is data.


In [ ]:
from pathlib import Path
import sys

COURSE_DIR = Path("curriculum/advanced/01-single-vs-multi-agent").resolve()
if str(COURSE_DIR) not in sys.path:
    sys.path.insert(0, str(COURSE_DIR))

from policy import (
    ArchitectureType, CoordinationLedger, ExecutionFixture, FailureCode,
    FailureEvent, PolicyError, RunStatus, WorkItem, aggregate_fixture,
    architecture_gate, detect_conflicts, pareto_front, resolve_partial_failure,
    validate_artifact, validate_topology,
)
from lab import (
    FIXED_TIME, REQUIRED_FACTS, agent_by_id, apply_untrusted_context_claim,
    build_agents, build_artifact, build_budget, build_conflicting_artifacts,
    build_evaluation_cases, build_handoff, build_northstar_case,
    build_evidence_registry, build_routing_cases, build_trusted_context,
    compare_all_architectures, compare_routers, completion_from_specialist_text,
    context_projection_experiment, deterministic_router, do_not_split_simple_faq,
    malicious_handoff_attempt, pipeline_not_team, run_architecture,
)

task = build_northstar_case()
trusted = build_trusted_context()
evidence = build_evidence_registry()
assert task.question.startswith("Why did EU checkout conversion fall")
assert set(task.required_evidence_ids) == set(evidence)
print(task.question)
print("required evidence:", task.required_evidence_ids)
print("production.execute authorized:", "production.execute" in trusted.authorization_capabilities)


The final line is intentionally `False`: the same architecture can investigate and propose without holding production authority. Agent names and prompts do not create a security boundary; capability policy, credentials, network controls, sandboxes, approval, and authorization do.


## Part 2 — Single-agent baseline

`SINGLE_GENERALIST` uses one owner, one scoped context, and all permitted read capabilities. This is the baseline to beat—not a strawman. We measure quality, tokens, latency, and maximum privileged-tool exposure.


In [ ]:
single = run_architecture(ArchitectureType.SINGLE_GENERALIST)
assert single.status is RunStatus.COMPLETED
assert single.metrics.required_evidence_recall == 1.0
print(single.model_dump_json(indent=2))


The baseline is grounded and complete. Any more complex architecture must demonstrate structural benefit while preserving these safety and evidence properties.


## Part 3 — Single agent with dynamic tools

Tool bloat does not automatically require an agent split. Tool search, dynamic loading, progressive disclosure, MCP discovery, skills, and programmatic tool calling can reduce model-visible surface while one owner remains in control. Tool visibility is still not authorization.


In [ ]:
dynamic = run_architecture(ArchitectureType.SINGLE_DYNAMIC_TOOLS)
print({
    "architecture": dynamic.architecture,
    "tokens": dynamic.metrics.total_tokens,
    "exposure": dynamic.metrics.privileged_tool_exposure,
    "baseline_tokens": single.metrics.total_tokens,
    "baseline_exposure": single.metrics.privileged_tool_exposure,
})
assert dynamic.metrics.total_tokens < single.metrics.total_tokens
assert dynamic.metrics.privileged_tool_exposure < single.metrics.privileged_tool_exposure


This smaller change lowers fixture token and exposure measures without introducing cross-agent coordination. It should be evaluated before a team.


## Part 4 — Coordination cost model

Total work is additive consumption; wall clock follows dependency batches. Within a conceptual parallel batch, wall clock increases by the maximum elapsed task time. A dependent batch starts afterward.


In [ ]:
def timed(name: str, elapsed_ms: int) -> WorkItem:
    return WorkItem(
        operation_id=name, model_work_ms=elapsed_ms, tool_work_ms=0,
        input_tokens=0, output_tokens=0, coordination_tokens=0,
        handoff_serialization_ms=0, cost_usd=0,
    )

timing = aggregate_fixture(ExecutionFixture(batches=(
    (timed("observability", 60), timed("deployment", 55), timed("customer", 80)),
    (timed("dependent-synthesis", 40),),
)))
print(timing.model_dump())
assert timing.total_work_ms == 235       # 195 parallel work + 40 dependent work
assert timing.wall_clock_latency_ms == 120  # 80 parallel wall clock + 40 dependent


Parallelism can lower elapsed latency while increasing aggregate compute and tool work. Production traces must additionally capture queueing, rate limits, contention, retries, and cancellation.


## Part 5 — Deterministic pipeline

Multiple LLM calls are not automatically multiple agents. A fixed `investigate → review evidence → synthesize` flow is a pipeline. Likewise, `generate → independent review → revise → deterministic gate` does not require autonomous debate.


In [ ]:
pipeline = run_architecture(ArchitectureType.PIPELINE)
assert pipeline_not_team() == "PIPELINE"
print({
    "verdict": pipeline_not_team(),
    "model_calls": pipeline.metrics.model_calls,
    "handoffs": pipeline.metrics.handoff_count,
    "completion_owner": pipeline.active_owner,
})


Use a pipeline when order and completion criteria are known. Open-ended producer/reviewer debate is justified only if a labelled evaluation shows enough benefit for the extra cost and failure surface.


## Part 6 — Manager plus specialists

The manager retains control and invokes observability, deployment, and customer-impact specialists for typed artifacts. This is equivalent to the “agents as tools” control model: specialists return bounded results; the manager owns synthesis.

![Supervisor delegates bounded work to three specialists, then validates evidence before synthesis](assets/team-topologies.svg)


In [ ]:
manager = run_architecture(ArchitectureType.MANAGER_SPECIALISTS)
assert manager.initial_owner == manager.active_owner == "manager"
assert manager.application_completed
assert completion_from_specialist_text("mission complete") is False
print({
    "owner": manager.active_owner,
    "validated_artifacts": manager.validated_artifact_ids,
    "specialist_completion_claim_accepted": False,
})


Completion is an application state, not a phrase a specialist may emit. The synthesizer consumes validated artifacts rather than raw hidden reasoning or unconstrained history.


## Part 7 — Handoff architecture

A handoff transfers active ownership. `HandoffEnvelope` binds source, target, tenant, task, required facts, artifacts, reason, depth, and deadline; `validate_topology()` checks the edge and capability attenuation first.


In [ ]:
handoff = run_architecture(ArchitectureType.HANDOFF)
assert handoff.initial_owner == "manager"
assert handoff.active_owner == "incident-specialist"
print({
    "before": handoff.initial_owner,
    "after": handoff.active_owner,
    "handoffs": handoff.metrics.handoff_count,
})


The owner change—not merely another model call—is the defining semantic. Manager delegation would have returned control to the manager instead.


## Part 8 — Parallel specialists

Observability, deployment, and customer-impact evidence are independent within this fixture and may fan out concurrently before synthesis. Concurrency is bounded by per-agent limits, the global budget, rate-limit groups, and shared dependencies.


In [ ]:
parallel = run_architecture(ArchitectureType.PARALLEL_SPECIALISTS)
print({
    "total_model_work_ms": parallel.metrics.total_model_work_ms,
    "total_tool_work_ms": parallel.metrics.total_tool_work_ms,
    "total_coordination_work_ms": parallel.metrics.total_coordination_work_ms,
    "total_work_ms": parallel.metrics.total_work_ms,
    "wall_clock_latency_ms": parallel.metrics.wall_clock_latency_ms,
    "critical_path_ms": parallel.metrics.critical_path_ms,
})
assert parallel.metrics.total_work_ms > single.metrics.total_work_ms
assert parallel.metrics.wall_clock_latency_ms < manager.metrics.wall_clock_latency_ms


The result demonstrates the intended distinction: more aggregate work can coexist with a shorter critical path. It does not claim that real fan-out is free.


## Part 9 — Typed artifacts, provenance, and state preservation

Specialist findings map to evidence IDs. Validation checks schema, task, tenant, agent identity, source provenance, evidence references, and capability scope before shared state or synthesis.


In [ ]:
artifact = build_artifact("deployment-specialist", ("deployment",))
validated = validate_artifact(
    artifact,
    task=task,
    trusted_context=trusted,
    agent=agent_by_id("deployment-specialist"),
    evidence_registry=evidence,
)
full_context, projected_context = context_projection_experiment()
print(validated.model_dump())
print(full_context.model_dump())
print(projected_context.model_dump())
assert projected_context.tokens < full_context.tokens
assert projected_context.sensitive_fields_exposed < full_context.sensitive_fields_exposed
assert projected_context.handoff_information_recall == 1.0


Task-scoped context preserves tenant, region, deploy ID, customer tier, and incident window while reducing disclosure. These are deterministic fixture measurements, not a claim that every projection is lossless.


## Part 10 — Routing dataset and comparison

Routing is set-valued: `SINGLE_ROUTE`, `MULTI_ROUTE`, `UNKNOWN`, or `AMBIGUOUS`. The dataset also includes an approval-gated production request. Frozen semantic/LLM/manager predictions make the metric calculation reproducible; they are not live benchmarks.


In [ ]:
for route_case in build_routing_cases():
    decision = deterministic_router(route_case)
    print(route_case.route_case_id, decision.state, decision.agent_ids, decision.approval_required)

router_metrics = compare_routers()
for router_name, metrics in router_metrics.items():
    print(router_name, metrics.model_dump())
assert router_metrics["deterministic-rule"].unknown_detection == 1.0


Exact route accuracy checks both state and destination set. Agent precision/recall expose over- and under-routing; unknown detection prevents forced delegation outside the supported domain.


## Part 11 — Failure, loop, and duplicate handling

The workflow models `TIMEOUT`, `AUTH_DENIED`, `POLICY_DENIED`, `INVALID_ARTIFACT`, and `SOURCE_UNAVAILABLE`. Required evidence—not agent count—determines whether partial failure permits degraded completion, abstention, or human review.


In [ ]:
failure = FailureEvent(code=FailureCode.TIMEOUT, agent_id="deployment-specialist", retryable=True)
covered = resolve_partial_failure(
    required_evidence_ids=task.required_evidence_ids,
    collected_evidence_ids=task.required_evidence_ids,
    failures=(failure,),
)
missing = resolve_partial_failure(
    required_evidence_ids=task.required_evidence_ids,
    collected_evidence_ids=tuple(x for x in task.required_evidence_ids if x != "deployment"),
    failures=(failure,),
)
ledger = CoordinationLedger()
call = dict(agent_id="deployment-specialist", task_id=task.case_id,
            inputs={"deploy_id": "deploy-1842"}, artifact_ids=())
ledger.record(**call)
try:
    ledger.record(**call)
except PolicyError as error:
    duplicate_code = str(error)
print(covered.status, missing.status, duplicate_code)
assert covered.status is RunStatus.COMPLETED_DEGRADED
assert missing.status is RunStatus.ABSTAINED
assert duplicate_code == "DUPLICATE_COORDINATION"


Invocation, handoff, depth, parallelism, cost, and deadline budgets bound the run. Topology validation also rejects repeated edges and cycles such as `A → B → A → B`.


## Part 12 — Security and capability boundaries

Delegated capability may not exceed the authorized parent/request intersection unless explicit application policy grants it. Trusted tenant, user, authorization, and incident values are immutable. An injected transfer instruction is treated as artifact data.


In [ ]:
same_context = apply_untrusted_context_claim(trusted, {"tenant_id": "globex", "user_id": "attacker"})
assert same_context is trusted

try:
    malicious_handoff_attempt()
except PolicyError as error:
    injection_result = str(error)

wrong_tenant = build_artifact("deployment-specialist", ("deployment",), tenant_id="globex")
try:
    validate_artifact(
        wrong_tenant, task=task, trusted_context=trusted,
        agent=agent_by_id("deployment-specialist"), evidence_registry=evidence,
    )
except PolicyError as error:
    tenant_result = str(error)

print({"injected_transfer": injection_result, "wrong_tenant": tenant_result})
assert injection_result == "UNAUTHORIZED_HANDOFF"
assert tenant_result == "ARTIFACT_TENANT_MISMATCH"


The policy denies control transfer even though the malicious text is syntactically valid data. Separate application credentials, network policy, sandboxing, approval, and authorization are still required in production.


## Part 13 — Same-workload evaluation

The evaluation set spans simple FAQ, single-domain incident, the cross-domain Northstar incident, security-sensitive request, multi-route investigation, and out-of-domain request. The architecture table below always uses the same Northstar case.


In [ ]:
evaluation_cases = build_evaluation_cases()
runs = compare_all_architectures()
assert len(evaluation_cases) == 6
assert {run.case_id for run in runs} == {task.case_id}
print("architecture                success tokens work wall cost exposure")
for run in runs:
    m = run.metrics
    print(f"{run.architecture.value:27} {m.task_success:.2f} {m.total_tokens:6} "
          f"{m.total_work_ms:4} {m.wall_clock_latency_ms:4} {m.cost_usd:.3f} "
          f"{m.privileged_tool_exposure}")


Interpret the fixture as a metric-semantics exercise: task success and exposure may improve while coordination tokens, cost, or total work worsen. Replace the fixture with trace-derived distributions before making a production decision.


## Part 14 — Quality gate and Pareto decision

A team is accepted only if success improves or privileged-tool exposure improves materially, grounding and safety do not regress, cost remains within budget, and latency remains within the SLO. Pareto analysis keeps non-dominated trade-offs instead of declaring one universal winner.


In [ ]:
parallel_gate = architecture_gate(
    single.metrics, parallel.metrics, max_cost_usd=0.03, latency_sla_ms=1_000
)
front = pareto_front(runs)
print("parallel-specialist gate:", parallel_gate.model_dump())
print("Pareto front:", [architecture.value for architecture in front])
print("simple FAQ:", do_not_split_simple_faq())
assert parallel_gate.accepted
assert do_not_split_simple_faq() == "KEEP_SINGLE"
assert len(front) > 1


The simple FAQ stays single. A cross-domain split can pass when its quality/exposure benefit pays for coordination and all guardrails hold. Different topologies remain preferable on different dimensions.


## Part 15 — Optional framework adapters

Choose primitives first: graph, manager, handoff, agents-as-tools, blackboard/shared state, message bus, or parallel fan-out.

- **LangGraph** is one graph/state-machine orchestration option; deterministic routing alone is not safety.
- **OpenAI Agents SDK** currently distinguishes manager-style `Agent.as_tool()` from handoffs that change the active agent. Swarm is a historical experimental predecessor.
- **AutoGen** and **CrewAI** are briefly mapped here; Advanced Courses 02 and 03 cover them deeply.

The next cell defines current OpenAI Agents SDK shapes but does not run them. It requires `openai-agents` and `OPENAI_API_KEY` only when a learner explicitly calls the function.


In [ ]:
async def optional_openai_agents_sdk_demo():
    from agents import Agent, Runner

    observability = Agent(
        name="Observability specialist",
        instructions="Return a bounded artifact from approved observability evidence.",
    )
    manager = Agent(
        name="Incident manager",
        instructions="Retain control and synthesize specialist artifacts.",
        tools=[observability.as_tool(
            tool_name="analyze_observability",
            tool_description="Analyze approved health and log evidence.",
        )],
    )
    triage = Agent(
        name="Incident triage",
        instructions="Hand off the bounded incident to the specialist.",
        handoffs=[observability],
    )
    manager_result = await Runner.run(manager, task.question)
    handoff_result = await Runner.run(triage, task.question)
    return manager_result.final_output, handoff_result.last_agent.name

print("Optional adapter defined; no credentialed call was made.")


### Production upgrade checklist

| Fixture boundary | Production control |
|---|---|
| Static evidence | Authenticated tenant-scoped sources, freshness, lineage, retention |
| Capability tuples | Central policy, short-lived credentials, network policy, sandboxes |
| In-process fan-out | Bounded queues, rate groups, cancellation, backpressure |
| Local validation | Schema registry, provenance attestation, audit trail |
| Point timings | Trace-derived latency/cost distributions and failure slices |
| No writes | Separate propose, approve, execute, idempotency, and receipts |

### Exercises

1. Add a rate-limit group and recompute the critical path.
2. Add a new evidence domain and test dynamic discovery before splitting.
3. Calibrate a semantic router on held-out labels.
4. Design a bounded reconciliation step for the conflict fixture.
5. Replace fixture metrics with sanitized production traces.

### Summary

- Multi-agent is an architecture choice, not a badge.
- Multiple model calls do not automatically mean multiple agents.
- Agent boundaries do not automatically create security boundaries.
- Parallel work increases total work without necessarily increasing wall-clock linearly.
- Handoff and manager delegation are different control models.
- A split is justified only when measured benefit exceeds coordination cost.
